# 02 — Data Preparation (Phase 1)

Canonical schema, validation, variable equivalence, temporal splits.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent) if Path.cwd().name=='notebooks' else str(Path.cwd()))
from src.utils.data_loader import DatasetSource
from src.schemas.canonical import META_COLUMNS, STATUS_TYPE_LABELS, NORMAL_STATUS_IDS
from src.validation.data_quality import basic_validation, timestamp_continuity, constant_columns, null_report
import pandas as pd

src = DatasetSource()
print('extracted:', src.extracted)
print('zip:', src.zip_path)


In [ ]:
# Load representative files
reps = {
    'A': src.read_csv('Wind Farm A/datasets/0.csv', sep=';'),
    'B': src.read_csv('Wind Farm B/datasets/2.csv', sep=';'),
    'C': src.read_csv('Wind Farm C/datasets/1.csv', sep=';'),
}
for k,v in reps.items():
    print(k, v.shape)


In [ ]:
# Validation report for representative files
for k,df in reps.items():
    bv = basic_validation(df)
    print('=== Farm', k)
    print('shape', bv['shape'])
    print('duplicates', bv['duplicates'])
    print('constant cols', bv['constant_columns'])
    print('inf', bv['inf_count'])
    print('null cols', len(bv['nulls']))


In [ ]:
# Timestamp continuity gaps
for k,df in reps.items():
    gaps = timestamp_continuity(df)
    print('Farm', k, 'gaps:', len(gaps))
    if len(gaps):
        print(gaps.head().to_string(index=False))


In [ ]:
# Variable equivalence: common meta + overlapping sensor base names
for k,df in reps.items():
    sensors = [c for c in df.columns if c not in META_COLUMNS]
    bases = sorted(set(c.rsplit('_',1)[0] for c in sensors))
    print('Farm', k, 'n sensor cols', len(sensors), 'n bases', len(bases))


In [ ]:
# Status type distribution
for k,df in reps.items():
    print('Farm', k, df['status_type_id'].value_counts().to_dict())


In [ ]:
# Save a small validation summary to interim (optional)
import json
out = Path('data/interim') if Path('data/interim').exists() else Path('../data/interim')
out.mkdir(parents=True, exist_ok=True)
summary = {k: {'shape': list(v.shape)} for k,v in reps.items()}
(out/'phase1_validation_summary.json').write_text(json.dumps(summary, indent=2))
print('saved', out/'phase1_validation_summary.json')
